# colab_10 — scGPT zero-shot embedding (regime #1 baseline)

Second notebook in the **foundation-model (FM) work**, the direct parallel to colab_09: it produces the **zero-shot** scGPT embedding of the same glia substrate — the un-adapted pretrained model's per-cell vectors, no fine-tuning. Zero-shot is regime #1 of the evaluation contract and does double duty — a competency deliverable ("embed with a second FM, inspect the space critically") **and** the frozen scGPT baseline that detector #1 (embedding drift) and evals #1–#2 measure every later scGPT CPT run against. The two FMs are always read together.

**The representational contrast is the point.** Geneformer (colab_09) rank-encodes **raw** counts — a cell becomes an ordered list of its most-expressed genes, magnitudes discarded. scGPT reads gene **symbols** against a fixed vocabulary and encodes **binned normalized expression** — each cell's genes carry a quantised expression level. Same cells, same niche, two different compressions of the transcriptome; the input difference is a biological hypothesis about what matters, not a formatting detail. Where colab_09 fed raw counts, this notebook applies scGPT's documented input pipeline (normalize to 1e4 + log1p, then the model's own per-cell binning) before embedding.

**Runtime.** scGPT is installed on Colab's **native Python** with a dependency-light install: the current scGPT source imports no scvi-tools, and guards torchtext / flash-attn / scib behind fallbacks, so the older <3.11 stack its packaging *declares* is not actually needed to load a checkpoint and embed. This replaces the earlier plan of a Python-3.10 runtime swap. The install is pinned to a specific scGPT commit for reproducibility.

The pipeline: (§1–§2) stand up + capture the scGPT environment and download the **whole-human** pretrained checkpoint; (§3) load the two label-carrying subset files (`micro_subset.h5ad` from colab_07, `astro_subset.h5ad` from colab_08 — raw counts, full ca. 26,514-gene panel, `substate` + `apoe_carrier` attached), combine them, and apply the scGPT input normalization; (§4) **vocab audit** — scGPT tokenizes by gene symbol over a fixed vocabulary, so intersect our panel and confirm the niche-critical genes survive, with **APOE out-of-vocab as a pre-registered hard fail** (it would make eval #2 impossible for this FM); (§5) embed; (§6) interrogate the raw embedding space with the same UMAP + silhouette battery as colab_09; (§7) save the baseline + audit trace.

Runs on a **GPU** runtime; high-RAM helpful — the embedding step materialises the in-vocab count matrix densely for the ca. 142k-cell object. No training happens here — that is colab_11+ (CPT).

## 1 — Setup

### 1a — Mount Drive, clone/pull the repo, install scGPT (native Python, `--no-deps`), download the whole-human checkpoint

Same Drive + repo bootstrap as the other notebooks, on Colab's native Python. scGPT is installed **from a pinned source commit with `--no-deps`** — its packaging declares an old, mutually-conflicting stack (scvi-tools<1.0, torchtext, an old scanpy), but the code actually loads and embeds without any of it, so those declared deps are skipped and only what the import chain really needs is added on top of Colab's base (`scanpy`, `anndata`, and `datasets` — the last only so `import scgpt` resolves; the embedding path never calls it). flash-attn is **not** installed; scGPT falls back to standard PyTorch attention (identical maths, slower). The **whole-human** checkpoint (vocab + config + weights, pretrained on 33M human cells) is downloaded to Drive so re-runs skip the download. Geneformer is deliberately **not** installed here — it has its own colab_09 environment.

In [ ]:
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/ad-glia-fm-prep"
os.makedirs(DRIVE_ROOT, exist_ok=True)

REPO_URL  = "https://github.com/pavlemic/ad-glia-fm-prep.git"
REPO_PATH = "/content/ad-glia-fm-prep"
if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

print("Python:", sys.version.split()[0])

# scGPT source only (--no-deps): its declared stack (scvi-tools<1.0, torchtext, old scanpy)
# is never imported by the load+embed path, so skip it and add only what the import chain
# needs on Colab's native base. Pinned to the CUDA-12.8 / FA2-compat commit.
SCGPT_COMMIT = "cebd6fae655b9c585a4807daa3ac31bb764f06b4"
!pip install --no-deps "git+https://github.com/bowang-lab/scGPT.git@{SCGPT_COMMIT}"
# Real runtime deps (scanpy, anndata, psutil, datasets==5.0.0) come from the requirements
# file; scGPT itself is the --no-deps git install above. flash-attn NOT installed (scGPT
# falls back to PyTorch attention). See requirements_scgpt.txt for the full rationale.
!pip install -r {REPO_PATH}/requirements_scgpt.txt

# --- whole-human pretrained checkpoint -> Drive (cached across runs) ---------------------
MODEL_DIR  = os.path.join(DRIVE_ROOT, "scgpt_whole_human")
CKPT_FILES = ["vocab.json", "args.json", "best_model.pt"]

def _have_ckpt(d):
    return all(os.path.exists(os.path.join(d, f)) for f in CKPT_FILES)

if not _have_ckpt(MODEL_DIR):
    os.makedirs(MODEL_DIR, exist_ok=True)
    WHOLE_HUMAN_FOLDER = "1oWh_-ZRdhtoGQ2Fw24HP41FgLoomVo-y"   # scGPT README pretrained table
    !pip install -q gdown
    import gdown
    gdown.download_folder(id=WHOLE_HUMAN_FOLDER, output=MODEL_DIR, quiet=False, use_cookies=False)
    # gdown may nest files in a subfolder — relocate MODEL_DIR to wherever best_model.pt landed.
    hits = [dp for dp, _, fs in os.walk(MODEL_DIR) if "best_model.pt" in fs]
    assert hits, f"best_model.pt not found under {MODEL_DIR} after download (Drive quota? download manually)"
    MODEL_DIR = hits[0]
else:
    print("checkpoint already present ->", MODEL_DIR)

assert _have_ckpt(MODEL_DIR), f"checkpoint incomplete in {MODEL_DIR}: need {CKPT_FILES}"
print("scGPT commit:", SCGPT_COMMIT[:7], "| checkpoint dir:", MODEL_DIR, "|", os.listdir(MODEL_DIR))

## 2 — Environment capture

### 2a — pip freeze + env JSON (records the resolved scGPT stack + commit)

`requirements_scgpt.txt` is deliberately dependency-light — scGPT is installed `--no-deps` from a pinned commit and the rest rides Colab's base — so this cell freezes exactly what got resolved, plus the scGPT commit, GPU, and CUDA, for the Methods record. `geneformer_version` reads `None` here by design (Geneformer is the colab_09 env). After a clean run the concrete versions are copied back into `requirements_scgpt.txt` and locked.

In [ ]:
import json, platform, subprocess, sys
from datetime import date

NOTEBOOK_ID = "colab_10"
TODAY = date.today().isoformat()
VERSIONS_DIR = os.path.join(REPO_PATH, "outputs", "software_versions")
os.makedirs(VERSIONS_DIR, exist_ok=True)

FREEZE_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_pip_freeze.txt")
!pip freeze > {FREEZE_PATH}

def _run(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, check=True).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None

def _ver(mod):
    # Actually import (not just check installed-metadata) so a broken import chain
    # surfaces here rather than silently reading as "not installed". geneformer is
    # expected to fail here by design (colab_09's env, not this one) -- everything
    # else failing is a real problem and gets printed, not just swallowed to None.
    try:
        m = __import__(mod)
    except Exception as e:
        print(f"  [_ver] import '{mod}' FAILED: {type(e).__name__}: {e}")
        return None
    try:
        return m.__version__
    except AttributeError:
        import importlib.metadata as ilm
        try:
            return ilm.version(mod)   # __version__ is deprecated on some scverse packages
        except Exception:
            return None

env_snapshot = {
    "notebook_id":    NOTEBOOK_ID,
    "date":           TODAY,
    "python_version": sys.version,
    "platform":       platform.platform(),
    "os_release":     platform.release(),
    "gpu":            _run(["nvidia-smi", "-L"]),
    "cuda":           _run(["nvcc", "--version"]),
    "git_commit":     _run(["git", "-C", REPO_PATH, "rev-parse", "HEAD"]),
    "scgpt_commit":       SCGPT_COMMIT,
    "scgpt_version":      _ver("scgpt"),
    "scanpy_version":     _ver("scanpy"),
    "anndata_version":    _ver("anndata"),
    "torch_version":      _ver("torch"),
    "numpy_version":      _ver("numpy"),
    "datasets_version":   _ver("datasets"),
    "geneformer_version": _ver("geneformer"),   # None here by design (Geneformer is colab_09)
    "model_checkpoint":   os.path.basename(MODEL_DIR),
}
ENV_JSON_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_env.json")
with open(ENV_JSON_PATH, "w") as f:
    json.dump(env_snapshot, f, indent=2)
print(json.dumps(env_snapshot, indent=2))

## 3 — Load the glia substrate

### 3a — Load both labelled subsets, combine, guard raw counts, apply the scGPT input normalization

Reads `micro_subset.h5ad` (colab_07) and `astro_subset.h5ad` (colab_08) — the post-QC-drop, substate-labelled halves of the glia set — and concatenates them into one ca. 142k-cell object, exactly as colab_09 did (same files, same gene panel, same `cell_index` key). Both descend from the same colab_05 glia object, so the gene panel is asserted identical before concatenation, and a `cell_index` is stamped for realigning embeddings afterward.

Where colab_09 stopped at raw counts (Geneformer rank-encodes raw), scGPT expects **normalized, log-transformed** expression as the input to its per-cell binning. So this cell first asserts `.X` is raw counts (guarding against a double-normalization), then applies scGPT's documented input transform — `normalize_total(target_sum=1e4)` then `log1p` — writing the result into `.X` and keeping the raw counts in `layers["counts"]`. The model's own binning (into expression-level bins) is applied downstream inside the embedding call, on top of this. Because scGPT's binning is per-cell rank-based this transform is close to rank-preserving, but applying it explicitly matches how the model was trained and how its tutorials use it — the defensible, reproducible choice rather than relying on that equivalence silently.

In [ ]:
import gc
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy.sparse as sp
import matplotlib.pyplot as plt

try:
    import psutil
    def _ram(tag):
        m = psutil.virtual_memory()
        print(f"[RAM] {tag:24s}: {m.used/1e9:5.1f} / {m.total/1e9:.1f} GB ({m.percent:.0f}%)")
except ImportError:
    def _ram(tag): pass

sc.settings.verbosity = 1
FIG_DIR = os.path.join(REPO_PATH, "figures", "colab_10")
os.makedirs(FIG_DIR, exist_ok=True)

MICRO_PATH = os.path.join(DRIVE_ROOT, "micro_subset", "micro_subset.h5ad")
ASTRO_PATH = os.path.join(DRIVE_ROOT, "astro_subset", "astro_subset.h5ad")
for p in (MICRO_PATH, ASTRO_PATH):
    if not os.path.exists(p):
        raise FileNotFoundError(f"missing labelled subset {p} (colab_07 / colab_08 output)")

micro = sc.read_h5ad(MICRO_PATH)
astro = sc.read_h5ad(ASTRO_PATH)
print("microglia subset:", micro.shape, "| substate:", micro.obs["substate"].value_counts().to_dict())
print("astrocyte subset:", astro.shape, "| substate:", astro.obs["substate"].value_counts().to_dict())

# Both subsets descend from the same colab_05 glia object -> identical gene panel (raw counts
# in .X). Assert before concatenating; a mismatch means one file was regenerated differently.
assert list(micro.var_names) == list(astro.var_names), "gene panels differ between subsets"

micro.obs["lineage"] = "microglia"
astro.obs["lineage"] = "astrocyte"
KEEP_OBS = ["lineage", "substate", "apoe_carrier", "study_id", "donor_id", "total_counts"]
micro.obs = micro.obs[[c for c in KEEP_OBS if c in micro.obs.columns]].copy()
astro.obs = astro.obs[[c for c in KEEP_OBS if c in astro.obs.columns]].copy()
glia = ad.concat([micro, astro], join="inner", index_unique="-")
del micro, astro; gc.collect()
glia.obs["cell_index"] = np.arange(glia.n_obs)   # stable key to realign embeddings after embed
glia.var["gene_name"]  = glia.var_names          # explicit symbol column for scGPT's gene_col
print("\ncombined glia:", glia.shape)
print("lineage:", glia.obs["lineage"].value_counts().to_dict())
print("apoe_carrier:", glia.obs["apoe_carrier"].value_counts(dropna=False).to_dict())

# raw-counts guard — scGPT's input transform must START from raw counts; a log-normalized
# .X here would be silently double-transformed.
_idx = np.random.default_rng(0).choice(glia.n_obs, size=min(2000, glia.n_obs), replace=False)
Xs = glia.X[_idx]
data = Xs.data if sp.issparse(Xs) else np.asarray(Xs).ravel()
frac_int = float(np.mean(np.mod(data, 1) == 0)) if data.size else 1.0
assert frac_int >= 0.99, f".X is not raw counts (int frac {frac_int:.3f}) — expected raw counts to normalize"
print("raw-counts int-frac:", round(frac_int, 3))

# scGPT input pipeline: keep raw in a layer, then normalize_total(1e4) + log1p into .X.
# The model's per-cell expression binning is applied downstream in 5a on top of this.
glia.layers["counts"] = glia.X.copy()
sc.pp.normalize_total(glia, target_sum=1e4)
sc.pp.log1p(glia)
print("applied normalize_total(1e4) + log1p; raw counts preserved in layers['counts']")
_ram("combined glia (normalized)")

## 4 — Vocabulary audit (setup audit A)

### 4a — Intersect the scGPT gene-symbol vocabulary, gate on APOE

scGPT tokenizes by **gene symbol** over a vocabulary fixed at pretraining (`vocab.json` in the checkpoint) — simpler than Geneformer's symbol→Ensembl step, but the same consequence: any measured gene outside the vocabulary is dropped from the cell's input. This cell loads the checkpoint vocabulary, counts how many of our ca. 26,514 symbols are in-vocab, then checks the niche-critical genes one by one (APOE / TREM2 / MS4A6A / CLU / GFAP / AQP4 / AIF1 / CSF1R). **APOE out-of-vocab is a pre-registered hard fail** — without it, eval #2 (the Stanton-core APOE axis) cannot be run for scGPT, so the notebook stops rather than produce an embedding that silently cannot answer the load-bearing question. The other niche genes are warn-only, logged to the audit report. The coverage is directly comparable to colab_09's Geneformer vocab audit — how each FM covers the same panel is itself a finding.

In [ ]:
from scgpt.tokenizer import GeneVocab

VOCAB_FILE = os.path.join(MODEL_DIR, "vocab.json")
vocab = GeneVocab.from_file(VOCAB_FILE)   # loads the checkpoint's own symbol->id map
vocab_size = len(vocab)
print("scGPT vocabulary size:", vocab_size)

# scGPT reads gene symbols directly (no Ensembl step). Intersect our panel with the vocab.
glia.var["in_scgpt_vocab"] = [s in vocab for s in glia.var["gene_name"]]
n_total = glia.n_vars
n_vocab = int(glia.var["in_scgpt_vocab"].sum())
print(f"\ngene panel: {n_total}")
print(f"  in scGPT vocab : {n_vocab}  ({n_vocab/n_total:.1%})")

# Niche-critical survival — the genes the whole niche rests on must be tokenizable.
NICHE_CRITICAL_GENES = ["APOE", "TREM2", "MS4A6A", "CLU", "GFAP", "AQP4", "AIF1", "CSF1R"]
panel = set(glia.var["gene_name"])
niche_status = {}
for g in NICHE_CRITICAL_GENES:
    niche_status[g] = {"in_panel": g in panel, "in_vocab": bool(g in vocab)}
print("\nniche-critical gene survival:")
for g, s in niche_status.items():
    flag = "OK" if s["in_vocab"] else ("WARN" if s["in_panel"] else "ABSENT")
    print(f"  {g:8s} panel={str(s['in_panel']):5} vocab={str(s['in_vocab']):5}  [{flag}]")

# Pre-registered gate: APOE out-of-vocab makes eval #2 (Stanton-core APOE axis) impossible
# for scGPT -> HARD FAIL. The other niche genes are warn-only (logged, not fatal).
assert niche_status["APOE"]["in_vocab"], (
    "APOE is not in the scGPT vocabulary — eval #2 cannot be run for this FM. "
    "Pre-registered hard fail (EVALUATION_CONTRACT eval #2)."
)
niche_warnings = [g for g, s in niche_status.items() if not s["in_vocab"]]
if niche_warnings:
    print("\nWARN — niche genes not in vocab (logged to audit):", niche_warnings)

VOCAB_AUDIT = {
    "vocab_size":      vocab_size,
    "n_genes_panel":   n_total,
    "n_in_vocab":      n_vocab,
    "frac_in_vocab":   round(n_vocab / n_total, 4),
    "niche_status":    niche_status,
    "niche_warnings":  niche_warnings,
    "apoe_hard_fail_gate": "passed",
}

## 5 — Embed

### 5a — Zero-shot cell embeddings from the pretrained whole-human model

Runs the forward pass through the **un-adapted** whole-human checkpoint and takes each cell's `<cls>`-token representation as its embedding — the zero-shot representation, regime #1. scGPT's `embed_data` does the whole input pipeline in one call: it subsets to in-vocab genes, tokenizes each cell as its (gene, binned-expression) pairs, truncates to the model's context length, runs the encoder, and L2-normalizes the output. `use_fast_transformer=False` forces the standard PyTorch attention path (flash-attn is not installed); the result is identical maths, just slower. The embedding is realigned to the combined object by `cell_index` (paranoia — the call preserves row order) and stored in `obsm["X_scGPT"]`. This is the direct counterpart of colab_09 §5b's `X_geneformer`.

In [ ]:
from scgpt.tasks import embed_data

# Lean input to keep RAM down: embed_data densifies the in-vocab matrix internally, so pass
# only .X + the gene-symbol column + cell_index (not the counts layer / full obs).
embed_input = ad.AnnData(
    X=glia.X.copy(),
    obs=glia.obs[["cell_index"]].copy(),
    var=pd.DataFrame({"gene_name": glia.var["gene_name"].values}, index=glia.var_names),
)

# embed_data: subset to in-vocab genes -> tokenize (gene, binned-expression) -> encoder ->
# L2-normalized <cls> embedding. Reads .X (normalized+log1p from 3a); binning happens inside.
# use_fast_transformer=False -> PyTorch attention; max_length=1200 = whole-human default context.
_ram("before embed")
embedded = embed_data(
    embed_input,
    MODEL_DIR,
    gene_col="gene_name",
    max_length=1200,
    batch_size=64,
    use_fast_transformer=False,
    return_new_adata=False,
)
_ram("after embed")

# Realign to the combined object by cell_index (row order is preserved — assert it).
assert embedded.n_obs == glia.n_obs, "embed_data changed the cell count — unexpected"
assert np.array_equal(embedded.obs["cell_index"].values, glia.obs["cell_index"].values), (
    "cell order changed during embedding — realign before use")
X_sc = np.asarray(embedded.obsm["X_scGPT"], dtype=np.float32)
glia.obsm["X_scGPT"] = X_sc
del embed_input, embedded; gc.collect()
print("X_scGPT:", X_sc.shape, "| emb dim:", X_sc.shape[1])

## 6 — Interrogate the zero-shot space

### 6a — UMAP + silhouettes: what does the raw FM embedding actually encode?

The critical-inspection step, run identically to colab_09 so the two FMs are directly comparable — the direct answer to the postdoc's "we don't know what vector space it is." A UMAP of the zero-shot embedding coloured by lineage, study, APOE-carrier, and substate, plus silhouette scores, ask three questions before any fine-tuning: does astro-vs-microglia **lineage** identity saturate the space (expected — any embedding separates the lineages); is **study** batch structure visible that the FM did not correct (FMs are claimed batch-robust — this tests it directly); and is there any **APOE** geometry within each lineage at all (colab_06's scVI latent and colab_09's Geneformer space both found none — this checks whether scGPT differs before CPT). The four colourings are shown as a grid **and** as individual full-size panels (fine structure is easier to read at full size). The silhouette battery uses the same helper, sampling, and APOE exclusion (`e2` off-contract) as colab_09, so the numbers line up column-for-column against the Geneformer baseline.

In [ ]:
from sklearn.metrics import silhouette_score

sc.pp.neighbors(glia, use_rep="X_scGPT", n_neighbors=15)
sc.tl.umap(glia)

UMAP_KEYS = ["lineage", "study_id", "apoe_carrier", "substate"]

# grid overview
fig, axes = plt.subplots(2, 2, figsize=(13, 11))
for ax, key in zip(axes.ravel(), UMAP_KEYS):
    sc.pl.umap(glia, color=key, ax=ax, show=False, frameon=False, size=4,
               title=f"scGPT zero-shot — {key}")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "6a_scgpt_zeroshot_umap_grid.png"), dpi=150, bbox_inches="tight")
plt.show()

# individual full-size panels (per-panel read is more reliable than a grid cell)
for key in UMAP_KEYS:
    figi, axi = plt.subplots(figsize=(8, 7))
    sc.pl.umap(glia, color=key, ax=axi, show=False, frameon=False, size=6,
               title=f"scGPT zero-shot — {key}")
    figi.tight_layout()
    figi.savefig(os.path.join(FIG_DIR, f"6a_scgpt_zeroshot_umap_{key}.png"), dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(figi)

# Quantify what the raw FM space encodes BEFORE any adaptation. Same helper as colab_09:
# apoe_carrier is E4-carrier vs non-carrier with E2-without-E4 EXCLUDED (off-contract third
# group), not scored as a class (EVALUATION_CONTRACT; colab_06 6b).
APOE_EXCLUDE_VALS = {"e2"}

def _sil(mask, label, exclude_vals=None):
    sub = glia[mask].copy()
    # non-in-place boolean AND: pandas Copy-on-Write can hand back a read-only array from
    # .values, and `keep &= ...` mutates in place -> ValueError on a CoW-protected array.
    # `keep = keep & ...` always builds a fresh array, so it's safe regardless of CoW state.
    keep = sub.obs[label].notna().values
    if exclude_vals:
        keep = keep & ~sub.obs[label].astype(str).isin(exclude_vals).values
    sub = sub[keep]
    y = sub.obs[label].astype(str).values
    if len(set(y)) < 2 or sub.n_obs < 50:
        return None
    n = min(10000, sub.n_obs)
    idx = np.random.default_rng(0).choice(sub.n_obs, n, replace=False)
    return round(float(silhouette_score(sub.obsm["X_scGPT"][idx], y[idx])), 4)

all_mask   = np.ones(glia.n_obs, bool)
micro_mask = (glia.obs["lineage"] == "microglia").values
astro_mask = (glia.obs["lineage"] == "astrocyte").values
sil = {
    "lineage_all":    _sil(all_mask,   "lineage"),    # expect HIGH — identity saturates
    "study_all":      _sil(all_mask,   "study_id"),   # batch the FM did NOT correct
    "apoe_micro":     _sil(micro_mask, "apoe_carrier", exclude_vals=APOE_EXCLUDE_VALS),
    "apoe_astro":     _sil(astro_mask, "apoe_carrier", exclude_vals=APOE_EXCLUDE_VALS),
    "substate_micro": _sil(micro_mask, "substate"),
    "substate_astro": _sil(astro_mask, "substate"),
}
print("silhouettes (zero-shot scGPT):")
for k, v in sil.items():
    print(f"  {k:16s}: {v}")

## 7 — Save + handoff

### 7a — Save the zero-shot baseline embedding, append audit trace, print commit commands

Writes a compact baseline artifact — per-cell embedding vectors plus the labels the evals slice on (no gene matrix, so it stays small) — to Drive as `glia_scgpt_zeroshot.h5ad`, the scGPT counterpart of colab_09's `glia_geneformer_zeroshot.h5ad`. This is the frozen reference every later scGPT CPT embedding is compared against (detector #1 drift; evals #1/#2), aligned by `cell_index`. The vocab-audit result and zero-shot silhouettes are appended to `outputs/audit_report.json` under `scgpt_zeroshot`, and the WSL commit/push commands are printed (Colab has no git credentials).

In [ ]:
import shlex

SCGPT_OUT_DIR = os.path.join(DRIVE_ROOT, "scgpt")
os.makedirs(SCGPT_OUT_DIR, exist_ok=True)

# Lean baseline artifact: per-cell zero-shot vectors + labels. detector #1 and evals #1/#2
# compare post-CPT embeddings against THIS, aligned by cell_index — compact (no gene matrix)
# but every label the evals slice on is kept.
emb_adata = ad.AnnData(
    X=glia.obsm["X_scGPT"],
    obs=glia.obs[["cell_index", "lineage", "substate", "apoe_carrier", "study_id", "donor_id"]].copy(),
)
emb_adata.obsm["X_umap"] = glia.obsm["X_umap"]
EMB_PATH = os.path.join(SCGPT_OUT_DIR, "glia_scgpt_zeroshot.h5ad")
emb_adata.write_h5ad(EMB_PATH)
print("saved zero-shot embedding ->", EMB_PATH, f"({os.path.getsize(EMB_PATH)/1e9:.2f} GB)")

AUDIT_REPORT_PATH = os.path.join(REPO_PATH, "outputs", "audit_report.json")
with open(AUDIT_REPORT_PATH) as f:
    report = json.load(f)
report["scgpt_zeroshot"] = {
    "status": "computed",
    "date": TODAY,
    "model_dir": os.path.basename(MODEL_DIR),
    "scgpt_commit": SCGPT_COMMIT,
    "n_cells": int(glia.n_obs),
    "emb_dim": int(glia.obsm["X_scGPT"].shape[1]),
    "vocab_audit": VOCAB_AUDIT,
    "zeroshot_silhouettes": sil,
    "embedding_file": os.path.relpath(EMB_PATH, DRIVE_ROOT),
}
with open(AUDIT_REPORT_PATH, "w") as f:
    json.dump(report, f, indent=2)
print("audit trace appended ->", AUDIT_REPORT_PATH)

rel = [os.path.relpath(p, REPO_PATH) for p in (FREEZE_PATH, ENV_JSON_PATH, AUDIT_REPORT_PATH)]
print("\n=== Commit + push (from WSL — Colab has no git creds) ===")
print("  cd /mnt/c/Users/micic/ad-glia-fm-prep && git add " + " ".join(shlex.quote(r) for r in rel))
print("  cd /mnt/c/Users/micic/ad-glia-fm-prep && git commit -m "
      "'colab_10: scGPT zero-shot embedding + vocab audit (regime #1 baseline)'")
print("  cd /mnt/c/Users/micic/ad-glia-fm-prep && git push")

### Carried forward

- **`glia_scgpt_zeroshot.h5ad`** (Drive) — the scGPT regime-#1 baseline embedding; colab_11+ (CPT) compares post-fine-tuning embeddings against it for detector #1 and evals #1–#2, aligned by `cell_index`. Read alongside `glia_geneformer_zeroshot.h5ad` — the two FMs are always reported together.
- **Resolved scGPT-env pins** (`outputs/software_versions/colab_10_*`) — copy the concrete versions back into `requirements_scgpt.txt`, lock the scGPT commit, and record that the native-Python `--no-deps` install replaced the Python-3.10 plan.
- **scGPT vs Geneformer, zero-shot** — vocab coverage (§4) and the silhouette battery (§6) are now available for both FMs on the same substrate; the first cross-FM read (does either encode APOE geometry before CPT? does either batch by study?) follows.